# CLP — Forecasting Pipeline Discovery via MCTS

**Run this notebook directly on Google Colab.** It clones both repos, loads GiftEval data via `kernels_playground/first_tests`, runs MCTS to discover the best forecasting pipelines, and visualises the results.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q numpy scikit-learn xgboost networkx matplotlib tqdm datasets

# Clone both repos
import os

if not os.path.isdir("graph_Time_series"):
    !git clone https://github.com/chahineNejm/graph_Time_series.git
else:
    print("graph_Time_series already cloned")

if not os.path.isdir("kernels_playground"):
    !git clone https://github.com/chahineNejm/kernels_playground.git
else:
    print("kernels_playground already cloned")

## 2. Imports

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Add repos to the path
for p in [".", "kernels_playground/first_tests"]:
    if p not in sys.path:
        sys.path.insert(0, p)

# CLP framework
from graph_Time_series import (
    State, Grammar, plot_grammar, print_mcts_tree,
    mcts_search, TOKEN_COLOURS
)
from graph_Time_series.tokens import register_all
from graph_Time_series.tokens.models import compute_mase

# Data loading from kernels_playground/first_tests
from utils.data import build_examples, prepare_examples
from utils.config import DATASETS

print("Imports OK")

## 3. Build grammar and visualise token graph

In [ ]:
grammar = Grammar()
register_all(grammar)
print(grammar)
print("Tokens:", grammar.token_names)

In [ ]:
plot_grammar(grammar, title="CLP Grammar — Token Transitions")

## 4. Load GiftEval data via kernels_playground

We use `build_examples()` and `prepare_examples()` from `kernels_playground/first_tests/utils/data.py` to load and normalise series from `Salesforce/GiftEvalParquet`.

In [ ]:
# Load raw examples
config_name = "electricity_H_long"
raw = build_examples(
    config=config_name,
    start=0,
    stop=25,        # max samples
    step=1,
    dataset_name=DATASETS["eval"],
)
print(f"Loaded {len(raw)} raw examples from {config_name}")

In [ ]:
# Prepare (z-normalise) and build history/future arrays
examples = prepare_examples(raw)

histories = [np.array(ex["history_value"], dtype=np.float32) for ex in examples]
futures   = [np.array(ex["future_value"],  dtype=np.float32) for ex in examples]

# Truncate to common lengths
min_hist = min(len(x) for x in histories)
min_fut  = min(len(x) for x in futures)

H = np.array([x[-min_hist:] for x in histories], dtype=np.float32)
F = np.array([x[:min_fut]   for x in futures],   dtype=np.float32)

print(f"History: {H.shape}  Future: {F.shape}")
data = {config_name: (H, F)}

In [ ]:
# ── Or load multiple configs ──
# Uncomment to run on several datasets:
#
# CONFIGS = ["electricity_H_long", "electricity_D_short", "traffic_weekly_short"]
#
# data = {}
# for cfg in tqdm(CONFIGS, desc="Loading"):
#     try:
#         raw = build_examples(cfg, start=0, stop=25, step=1, dataset_name=DATASETS["eval"])
#         exs = prepare_examples(raw)
#         hists = [np.array(e["history_value"], dtype=np.float32) for e in exs]
#         futs  = [np.array(e["future_value"],  dtype=np.float32) for e in exs]
#         mh, mf = min(len(x) for x in hists), min(len(x) for x in futs)
#         data[cfg] = (
#             np.array([x[-mh:] for x in hists], dtype=np.float32),
#             np.array([x[:mf]  for x in futs],   dtype=np.float32),
#         )
#         print(f"  {cfg}: {data[cfg][0].shape}")
#     except Exception as e:
#         print(f"  SKIP {cfg}: {e}")
# print(f"\nLoaded {len(data)} configs")

## 5. Run MCTS search per config

Each config gets its own MCTS search. The tqdm bars show progress at every level: configs, MCTS iterations, and LOO fits inside models.

In [ ]:
N_ITERATIONS = 40   # increase for deeper search
PUCT_C = 1.5        # exploration constant

results = {}

for cfg, (H, F) in tqdm(data.items(), desc="Configs", total=len(data)):
    print(f"\n{'='*70}")
    print(f"Config: {cfg}  ({H.shape[0]} samples)")
    print(f"{'='*70}")

    state = State(H, F)
    res = mcts_search(grammar, state,
                      n_iterations=N_ITERATIONS,
                      puct_c=PUCT_C,
                      verbose=True)

    results[cfg] = res
    print(f"\n  >>> Best: {res['best_chain']}  (MASE={res['best_mase']:.4f})")

## 6. Summary table

In [ ]:
print(f"{'Config':<40s}  {'Best chain':<55s}  {'MASE':>8s}")
print("-" * 105)
for cfg, res in results.items():
    print(f"{cfg:<40s}  {res['best_chain']:<55s}  {res['best_mase']:8.4f}")

## 7. MCTS tree visualisation

In [ ]:
for cfg, res in results.items():
    print(f"\n{'='*60}")
    print(f"  {cfg}")
    print(f"{'='*60}")
    print_mcts_tree(res["root"], max_depth=4)

## 8. Convergence plots

In [ ]:
if not results:
    print("No results to plot.")
else:
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False)

    for ax, (cfg, res) in zip(axes[0], results.items()):
        mases = [h["mase"] for h in res["history"]]
        best_so_far = np.minimum.accumulate(mases)
        ax.plot(mases, alpha=0.3, label="per-iteration")
        ax.plot(best_so_far, linewidth=2, label="best so far")
        ax.set_xlabel("Iteration")
        ax.set_ylabel("MASE")
        ax.set_title(cfg.split("_")[0])
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 9. Replay best chain (transformation log)

In [ ]:
cfg_name = list(results.keys())[0]
H, F = data[cfg_name]
best = results[cfg_name]

chain_tokens = best["best_chain"].replace(" -> ", ",").split(",")
chain_tokens = [t.strip() for t in chain_tokens if t.strip() != "STOP"]

print(f"Replaying: {best['best_chain']}  on  {cfg_name}\n")

state = State(H, F)
for tok_name in chain_tokens:
    token = grammar.tokens[tok_name]
    state = token.apply(state)

grammar.tokens["STOP"].apply(state)

state.print_log()
print(f"\nFinal MASE: {state.mase:.4f}")

## 10. Forecast visualisation

In [ ]:
forecast = state.features["final_forecast"]
n_show = min(4, state.n_samples)
fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 3))
if n_show == 1:
    axes = [axes]

rng = np.random.default_rng(42)
idxs = rng.choice(state.n_samples, size=n_show, replace=False)

for ax, i in zip(axes, idxs):
    hist = state.original_history[i]
    fut  = state.original_future[i]
    pred = forecast[i]
    t_h = np.arange(len(hist))
    t_f = np.arange(len(hist), len(hist) + len(fut))

    ax.plot(t_h, hist, "k-", label="history")
    ax.plot(t_f, fut, "b-", linewidth=2, label="actual")
    ax.plot(t_f, pred, "r--", linewidth=2, label="forecast")
    ax.set_title(f"Sample {i}")
    ax.legend(fontsize=7)

plt.suptitle(f"{cfg_name} — {best['best_chain']}", fontsize=10)
plt.tight_layout()
plt.show()

## 11. Example: add your own token

In [ ]:
from graph_Time_series.token import Token, _shapes

class CleanZScore(Token):
    name = "zscore"
    token_class = "cleaning"
    reads = ["raw_history"]
    writes = ["cleaned"]
    description = "Per-sample z-score normalisation"

    def apply(self, state):
        X = state.features["raw_history"]
        mu = X.mean(axis=1, keepdims=True)
        sigma = X.std(axis=1, keepdims=True) + 1e-8
        state.features["cleaned"] = (X - mu) / sigma
        state.log_step(self.name,
                       _shapes(state, self.reads),
                       {"cleaned": state.features["cleaned"].shape})
        state.token_sequence.append(self.name)
        return state

grammar.register(CleanZScore(), follows=["START"])

from graph_Time_series.tokens.features import FeatRaw, FeatFFTEncode
grammar.register(FeatRaw(),       follows=["zscore"], leads_to=[])
grammar.register(FeatFFTEncode(), follows=["zscore"], leads_to=[])

print(f"Grammar now has {grammar.n_tokens} tokens")
print("Valid after START:", [n for n in grammar.graph.successors("START")])

In [ ]:
# Run MCTS again with the new token available
cfg_name = list(data.keys())[0]
H, F = data[cfg_name]

state = State(H, F)
res2 = mcts_search(grammar, state, n_iterations=40, puct_c=1.5, verbose=False)
print(f"Best: {res2['best_chain']}  (MASE={res2['best_mase']:.4f})")